In [17]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from scipy.signal import find_peaks, welch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

In [5]:
import sys
sys.path.append("..")
from UCI.pre_processing import  load_UCI_dataset

In [6]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 8640
Validation recordings: 960
Test recordings: 2400


100%|██████████| 8640/8640 [01:36<00:00, 89.57it/s] 


Skipped recordings: 4


100%|██████████| 960/960 [00:10<00:00, 87.40it/s] 


Skipped recordings: 0


100%|██████████| 2400/2400 [00:27<00:00, 86.68it/s] 


Skipped recordings: 1


In [7]:
X_train = X_train.numpy()
X_val = X_val.numpy()
X_test = X_test.numpy()

y_train = y_train.numpy()
y_val = y_val.numpy()
y_test = y_test.numpy()

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (468372, 1, 1000), y_train shape: (468372, 2)


## Feature Exraction ##

In [8]:
from scipy.stats import skew, kurtosis
from scipy.signal import welch

In [9]:
def extract_ppg_features(ppg):

    return {
        "mean": np.mean(ppg),
        "std": np.std(ppg),
        "skewness": skew(ppg),
        "kurtosis": kurtosis(ppg),
        "rms": np.sqrt(np.mean(ppg ** 2)),
        "max": np.max(ppg),
        "min": np.min(ppg),
        "range": np.ptp(ppg)
    }

In [10]:
def extract_peak_features(ppg, fs=125):

    peaks, _ = find_peaks(
        ppg,
        distance=50,
        prominence=0.1 * np.std(ppg)
    )

    if len(peaks) < 2:

        return {
            "heart_rate": np.nan,
            "mean_peak_amplitude": np.nan,
            "std_peak_amplitude": np.nan,
            "mean_peak_distance": np.nan
        }

    peak_amplitudes = ppg[peaks]

    peak_distances = np.diff(peaks) / fs

    mean_peak_distance = np.mean(
        peak_distances
    )

    heart_rate = 60 / mean_peak_distance

    return {
        "heart_rate": heart_rate,
        "mean_peak_amplitude": np.mean(
            peak_amplitudes
        ),
        "std_peak_amplitude": np.std(
            peak_amplitudes
        ),
        "mean_peak_distance": mean_peak_distance
    }

In [11]:
def extract_vpg_apg_features(ppg, fs=125):

    vpg = np.gradient(ppg) * fs

    apg = np.gradient(vpg) * fs

    features = {

        "max_vpg": np.max(vpg),
        "min_vpg": np.min(vpg),
        "mean_vpg": np.mean(vpg),
        "std_vpg": np.std(vpg),

        "max_apg": np.max(apg),
        "min_apg": np.min(apg),
        "mean_apg": np.mean(apg),
        "std_apg": np.std(apg)
    }

    return features

In [12]:
def extract_frequency_features(ppg, fs=125):

    frequencies, power = welch(
        ppg,
        fs=fs,
        nperseg=min(256, len(ppg))
    )

    valid = frequencies > 0

    frequencies = frequencies[valid]
    power = power[valid]

    dominant_frequency = frequencies[
        np.argmax(power)
    ]

    spectral_energy = np.trapz(
        power,
        frequencies
    )

    return {
        "dominant_frequency": dominant_frequency,
        "spectral_energy": spectral_energy
    }

In [13]:
def extract_22_features(ppg, fs=125):

    features = {}

    features.update(
        extract_ppg_features(ppg)
    )

    features.update(
        extract_peak_features(ppg, fs)
    )

    features.update(
        extract_vpg_apg_features(ppg, fs)
    )

    features.update(
        extract_frequency_features(ppg, fs)
    )

    return features

In [14]:
test_window = X_train[0, 0, :]

features = extract_22_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

Number of features: 22
{'mean': np.float32(-0.08624366), 'std': np.float32(0.9701424), 'skewness': np.float64(0.7124735116958618), 'kurtosis': np.float32(-0.67300916), 'rms': np.float32(0.9739683), 'max': np.float32(2.1523602), 'min': np.float32(-1.3754413), 'range': np.float32(3.5278015), 'heart_rate': np.float64(86.70520231213874), 'mean_peak_amplitude': np.float32(1.8730375), 'std_peak_amplitude': np.float32(0.11326101), 'mean_peak_distance': np.float64(0.692), 'max_vpg': np.float32(26.923552), 'min_vpg': np.float32(-18.560934), 'mean_vpg': np.float32(0.008362579), 'std_vpg': np.float32(11.13607), 'max_apg': np.float32(605.525), 'min_apg': np.float32(-599.1514), 'mean_apg': np.float32(2.167143), 'std_apg': np.float32(194.18651), 'dominant_frequency': np.float64(1.46484375), 'spectral_energy': np.float64(0.9716021102240427)}


In [15]:
def extract_features_from_dataset(
    X,
    y,
    fs=125
):

    rows = []

    for i in tqdm(range(len(X))):

        ppg = X[i, 0, :]

        features = extract_22_features(
            ppg,
            fs
        )

        features["SBP"] = y[i, 0]
        features["DBP"] = y[i, 1]

        rows.append(features)

    return pd.DataFrame(rows)

In [16]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

NameError: name 'tqdm' is not defined